In [1]:
import requests
import urllib3
urllib3.disable_warnings()

api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={
        "page":1,
        "page_size":50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [2]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [3]:
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [4]:
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page":page,
            "page_size":50
        },
        timeout=10
    )

    page_data = r.json()

    devices.extend(page_data["items"])
    # TODO

print("device count:", len(devices))

device count: 5000


In [7]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [8]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


In [9]:
card = cards[0]

In [10]:
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [11]:
company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip()) 

print(company_id, name, status, industry, zone, device_count)

C000001 临港精工001有限公司 运营中 汽车零部件 智造一区 6


In [12]:
companies = []   # 第一页的企业列表

for card in cards:
    # 1) dt/dd 转字典
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    # 2) 提取各字段
    companies.append({
        "company_id": card.get("data-company-id"),          # 来自 data-* 属性
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),
    })

print("本页企业数：", len(companies))


本页企业数： 20


In [13]:
def find_next_link(soup):
    """在分页栏里精确找到文本为"下一页"的 <a>，避开"上一页" """
    for a in soup.select("nav.pagination a"):
        if a.get_text(strip=True) == "下一页":
            return a
    return None

next_link = find_next_link(soup)

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))


下一页
/companies?page=2


In [14]:
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)


https://115.159.159.82/companies?page=2


In [15]:
companies = [] 
url = base_url + "/companies"  
page_counts = []

for page_no in range(1, 4): 
    resp = requests.get(url, timeout=10, verify=False)
    page_soup = BeautifulSoup(resp.text, "html.parser")

    page_cards = page_soup.select("article.company-card")
    for card in page_cards:
        details = {}
        for item in card.select("dl > div"):
            key = item.select_one("dt").get_text(strip=True)
            value = item.select_one("dd").get_text(strip=True)
            details[key] = value
        companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台", "").strip()),
            "status": card.select_one(".badge").get_text(strip=True),
        })
    page_counts.append(len(page_cards))
    print(f"第 {page_no} 页：{len(page_cards)} 条")

    nxt = find_next_link(page_soup)
    if nxt is None:
        print("没有下一页，提前结束")
        break
    url = urljoin(url, nxt.get("href"))

print()
print("Company 总数：", len(companies))
print("唯一 company_id 数：", len(set(c["company_id"] for c in companies)))


第 1 页：20 条
第 2 页：20 条
第 3 页：20 条

Company 总数： 60
唯一 company_id 数： 60


In [16]:
print("companies:", len(companies))
print(
    "unique company_id:",
    len(set(x["company_id"] for x in companies))
)

print("devices:", len(devices))
print(
    "unique device_id:",
    len(set(x["device_id"] for x in devices))
)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


In [17]:
from collections import Counter

device_counts = Counter(
    x["company_id"] for x in devices
)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]          
    actual = device_counts.get(cid, 0)       
    if expected != actual:
        mismatch_count += 1

In [18]:
print("Company 总数:", len(companies))
print("唯一 company_id 数:", len(set(c["company_id"] for c in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len(set(d["device_id"] for d in devices)))
print("Company-Device 不一致企业数:",mismatch_count )

Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company-Device 不一致企业数: 0
